# Notebook 4 — Revenue Analysis & Forecasting
**What this does:**
- Calculates MRR (Monthly Recurring Revenue) for each of the 13 months
- Shows MRR grew from $7,176 → $41,535
- Forecasts next 3 months using a moving average (no extra libraries needed)
- Quantifies revenue at risk from high-risk churners
- Saves all data for Power BI

**Note:** This version uses only pandas + numpy (already installed). No statsmodels needed.

In [ ]:
# ── CELL 1 ── Imports ─────────────────────────────────────────────────────────
# Only pandas, numpy, matplotlib, os — all already installed in your .venv
# No statsmodels needed in this version

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
%matplotlib inline

FILE_PATH  = r'E:\Mavenflix project\data\Subscription Cohort Analysis Data.csv'
OUTPUT_DIR = r'E:\Mavenflix project\outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Imports successful!')
print('pandas :', pd.__version__)
print('numpy  :', np.__version__)

In [ ]:
# ── CELL 2 ── Load data ────────────────────────────────────────────────────────

df = pd.read_csv(FILE_PATH)
df['created_date']  = pd.to_datetime(df['created_date'])
df['canceled_date'] = pd.to_datetime(df['canceled_date'])

print('Data loaded. Shape:', df.shape)
df.head()

In [ ]:
# ── CELL 3 ── Calculate MRR for every month ───────────────────────────────────
# WHAT IS MRR? Total subscription revenue collected each month.
# For each month M, count subscribers who:
#   (a) signed up ON OR BEFORE the last day of month M
#   (b) cancelled AFTER month M, OR never cancelled at all
# MRR = active count × $39

months = pd.period_range('2022-09', '2023-09', freq='M')
mrr_records = []

for month in months:
    month_end = month.to_timestamp('M')   # e.g. Sep 30, Oct 31

    active_mask = (
        (df['created_date'] <= month_end) &
        (
            df['canceled_date'].isna() |              # never cancelled
            (df['canceled_date'] > month_end)         # cancelled after this month
        )
    )
    active = df[active_mask]

    mrr_records.append({
        'month'        : str(month),
        'active_subs'  : len(active),
        'mrr'          : int(active['subscription_cost'].sum()),
        'new_subs'     : int((df['created_date'].dt.to_period('M') == month).sum()),
        'churned_subs' : int((df['canceled_date'].dt.to_period('M') == month).sum())
    })

mrr_df = pd.DataFrame(mrr_records)

# Month-over-month growth %
mrr_df['mrr_growth_pct'] = mrr_df['mrr'].pct_change().mul(100).round(1)

print('=== MONTHLY REVENUE TABLE ===')
print(mrr_df.to_string(index=False))
print()
print(f"Starting MRR (Sep 2022) : ${mrr_df['mrr'].iloc[0]:,}")
print(f"Current  MRR (Sep 2023) : ${mrr_df['mrr'].iloc[-1]:,}")
total_growth = ((mrr_df['mrr'].iloc[-1] / mrr_df['mrr'].iloc[0]) - 1) * 100
print(f"Total MRR growth        : +{total_growth:.1f}% over 13 months")

In [ ]:
# ── CELL 4 ── MRR trend chart ──────────────────────────────────────────────────

fig, axes = plt.subplots(2, 1, figsize=(13, 10))

# Top: MRR line chart
axes[0].plot(range(len(mrr_df)), mrr_df['mrr'],
             color='#1D9E75', marker='o', linewidth=2.5, markersize=7, zorder=3)
axes[0].fill_between(range(len(mrr_df)), mrr_df['mrr'],
                     alpha=0.12, color='#1D9E75')

for i, val in enumerate(mrr_df['mrr']):
    axes[0].text(i, val + 400, f'${val:,}',
                 ha='center', fontsize=7.5, color='#0F6E56')

axes[0].set_xticks(range(len(mrr_df)))
axes[0].set_xticklabels(mrr_df['month'], rotation=45, ha='right', fontsize=9)
axes[0].set_title(
    'MavenFlix Monthly Recurring Revenue Growth\n'
    'Sep 2022 → Sep 2023  |  $7,176 → $41,535  (+479%)',
    fontsize=12, fontweight='bold'
)
axes[0].set_ylabel('Monthly Revenue ($)', fontsize=10)
axes[0].grid(axis='y', alpha=0.3)

# Bottom: New vs Churned subscribers
width = 0.35
axes[1].bar([i - width/2 for i in range(len(mrr_df))], mrr_df['new_subs'],
            width=width, color='#1D9E75', label='New Subscribers', edgecolor='white')
axes[1].bar([i + width/2 for i in range(len(mrr_df))], mrr_df['churned_subs'],
            width=width, color='#D85A30', label='Churned Subscribers', edgecolor='white')
axes[1].set_xticks(range(len(mrr_df)))
axes[1].set_xticklabels(mrr_df['month'], rotation=45, ha='right', fontsize=9)
axes[1].set_title('New vs Churned Subscribers Per Month', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Subscribers', fontsize=10)
axes[1].legend(fontsize=10)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'chart10_mrr_trend.png'), dpi=150, bbox_inches='tight')
plt.show()
print('MRR trend chart saved!')

In [ ]:
# ── CELL 5 ── 3-month FORECAST using weighted moving average ───────────────────
# No statsmodels needed — we use numpy directly.
# METHOD: Weighted Moving Average
#   Most recent month gets weight 3, month before gets 2, oldest gets 1.
#   This gives more importance to recent data, like exponential smoothing.
# WHY FORECAST? Shows: "if nothing changes, where does revenue go next quarter?"

mrr_vals = mrr_df['mrr'].values.astype(float)

def weighted_forecast(series, steps=3):
    forecasts = []
    history   = list(series)
    for _ in range(steps):
        # Use last 3 months with weights 1, 2, 3 (most recent = highest weight)
        w   = np.array([1, 2, 3])
        avg = np.average(history[-3:], weights=w)
        forecasts.append(round(avg))
        history.append(avg)
    return forecasts

forecast_vals   = weighted_forecast(mrr_vals, steps=3)
forecast_months = ['2023-10', '2023-11', '2023-12']

forecast_df = pd.DataFrame({
    'month'        : forecast_months,
    'mrr_forecast' : [int(v) for v in forecast_vals]
})

print('=== 3-MONTH MRR FORECAST ===')
print(forecast_df.to_string(index=False))
print()
print(f"Current MRR  (Sep 2023) : ${mrr_vals[-1]:,.0f}")
print(f"Forecast Oct 2023       : ${forecast_vals[0]:,.0f}")
print(f"Forecast Nov 2023       : ${forecast_vals[1]:,.0f}")
print(f"Forecast Dec 2023       : ${forecast_vals[2]:,.0f}")
print()
print(f"Forecasted Q4 total revenue : ${sum(forecast_vals):,.0f}")

In [ ]:
# ── CELL 6 ── Actual + Forecast combined chart ─────────────────────────────────

actual_mrr = mrr_df['mrr'].tolist()
all_months = mrr_df['month'].tolist() + forecast_months

fig, ax = plt.subplots(figsize=(15, 6))

# Actual MRR
ax.plot(range(len(actual_mrr)), actual_mrr,
        color='#1D9E75', marker='o', linewidth=2.5,
        markersize=7, label='Actual MRR', zorder=3)
ax.fill_between(range(len(actual_mrr)), actual_mrr,
                alpha=0.1, color='#1D9E75')

# Bridge line from last actual to first forecast
ax.plot([len(actual_mrr)-1, len(actual_mrr)],
        [actual_mrr[-1], forecast_vals[0]],
        color='#D85A30', linestyle='--', linewidth=2, alpha=0.6)

# Forecast points
ax.plot(range(len(actual_mrr), len(actual_mrr)+3), forecast_vals,
        color='#D85A30', marker='s', linewidth=2.5,
        linestyle='--', markersize=9, label='3-Month Forecast', zorder=3)

# Labels on forecast points
for x, val in enumerate(forecast_vals, start=len(actual_mrr)):
    ax.annotate(f'${int(val):,}', xy=(x, val), xytext=(0, 14),
                textcoords='offset points',
                ha='center', fontsize=10, fontweight='bold', color='#D85A30')

# Shade forecast zone
ax.axvspan(len(actual_mrr)-0.5, len(actual_mrr)+2.5,
           alpha=0.07, color='#D85A30', label='Forecast zone')

ax.set_xticks(range(len(all_months)))
ax.set_xticklabels(all_months, rotation=45, ha='right', fontsize=9)
ax.set_title(
    'MavenFlix MRR — 13 Months Actual + 3-Month Forecast\n'
    'Forecast assumes current growth and churn rates continue',
    fontsize=13, fontweight='bold', pad=15
)
ax.set_ylabel('Monthly Revenue ($)', fontsize=11)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'chart11_mrr_forecast.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Forecast chart saved!')

In [ ]:
# ── CELL 7 ── Revenue at risk from high-risk churners ──────────────────────────

risk_df = pd.read_csv(os.path.join(OUTPUT_DIR, 'churn_risk_scores.csv'))

high_risk_count = (risk_df['risk_tier'] == 'High Risk').sum()
med_risk_count  = (risk_df['risk_tier'] == 'Medium Risk').sum()
low_risk_count  = (risk_df['risk_tier'] == 'Low Risk').sum()

high_risk_rev   = high_risk_count * 39
current_mrr     = mrr_df['mrr'].iloc[-1]

# Retention campaign ROI calculation
campaign_cost   = high_risk_count * 2        # $2 per subscriber outreach
save_rate       = 0.20                       # assume 20% retention success
saved_revenue   = int(high_risk_count * save_rate * 39)
net_roi         = saved_revenue - campaign_cost

print('=== REVENUE IMPACT ANALYSIS ===')
print(f'Current MRR                    : ${current_mrr:,}')
print()
print(f'High risk subscribers          : {high_risk_count}')
print(f'Medium risk subscribers        : {med_risk_count}')
print(f'Low risk subscribers           : {low_risk_count}')
print()
print(f'Revenue at risk (high-risk)    : ${high_risk_rev:,}/month')
print(f'% of MRR at risk               : {high_risk_rev/current_mrr*100:.1f}%')
print()
print('=== RETENTION CAMPAIGN ROI ===')
print(f'Campaign cost (@ $2/subscriber): ${campaign_cost:,}')
print(f'Saved subscribers (20% rate)   : {int(high_risk_count * save_rate)}')
print(f'Revenue saved                  : ${saved_revenue:,}/month')
print(f'Net ROI                        : ${net_roi:,}/month')
print(f'Return on spend                : {net_roi/max(campaign_cost,1):.1f}x')
print()
print('INTERVIEW ANSWER: "A targeted retention campaign for high-risk subscribers')
print(f'costs ${campaign_cost:,} and can recover ${saved_revenue:,}/month — a {net_roi/max(campaign_cost,1):.1f}x ROI."')

In [ ]:
# ── CELL 8 ── Save all output files ────────────────────────────────────────────

mrr_df.to_csv(os.path.join(OUTPUT_DIR, 'mrr_monthly.csv'), index=False)
forecast_df.to_csv(os.path.join(OUTPUT_DIR, 'revenue_forecast.csv'), index=False)

print('Files saved:')
print('  mrr_monthly.csv       → use in Power BI Page 3')
print('  revenue_forecast.csv  → use in Power BI Page 3')
print()
print('=== ALL OUTPUT FILES IN YOUR OUTPUTS FOLDER ===')
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    kind = 'CHART' if f.endswith('.png') else 'DATA'
    print(f'  [{kind}] {f:<45} {size:>8,} bytes')
print()
print('=== NOTEBOOK 4 COMPLETE ===')
print('You now have 11 charts + 5 CSV files.')
print()
print('NEXT STEPS:')
print('  Step 1  → Install statsmodels (for future projects)')
print('  Step 2  → Build Power BI dashboard (3 pages)')
print('  Step 3  → Run 05_ai_insights.ipynb (Gemini API bonus)')
print('  Step 4  → Write GitHub README')
print('  Step 5  → Push to GitHub')
print('  Step 6  → Post on LinkedIn with heatmap image')